In [1]:
from pymongo import MongoClient

cliente = MongoClient("mongodb://localhost:27017/")

print(cliente.list_database_names())

['admin', 'config', 'local']


In [2]:
db = cliente["RecursosHumanos"]
coleccion = db["Curriculums"]

print("Base de datos:", db.name)
print("Colección:", coleccion.name)

Base de datos: RecursosHumanos
Colección: Curriculums


In [3]:
import json

with open("data/curriculumns.json", "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

resultado = coleccion.insert_many(datos)

print("Documentos insertados:", len(resultado.inserted_ids))

Documentos insertados: 3


In [4]:
for curriculum in coleccion.find():
    print(curriculum)
    print("-" * 80)


{'_id': ObjectId('6ac7b1553ea1757fb660bdd8'), 'nombre': 'Bill Gates', 'titular': 'Greater Seattle Area | Philanthropy', 'resumen': 'Co-chair of the Bill & Melinda Gates Foundation. Chairman, Microsoft Corporation.', 'experiencia': [{'puesto': 'Co-chair', 'empresa': 'Bill & Melinda Gates Foundation', 'inicio': 2000, 'fin': 'Presente'}, {'puesto': 'Co-founder, Chairman', 'empresa': 'Microsoft', 'inicio': 1975, 'fin': 'Presente'}], 'formacion': [{'institucion': 'Harvard University', 'inicio': 1973, 'fin': 1975}, {'institucion': 'Lakeside School, Seattle', 'inicio': 1967, 'fin': 1973}], 'contacto': {'blog': 'thegatesnotes.com', 'twitter': '@BillGates'}}
--------------------------------------------------------------------------------
{'_id': ObjectId('6ac7b1553ea1757fb660bdd9'), 'nombre': 'Steve Wozniak', 'titular': 'Silicon Valley | Hardware & Computing', 'resumen': 'Co-founder of Apple Computer. Designer of Apple I and Apple II.', 'experiencia': [{'puesto': 'Co-founder & Chief Engineer', 

In [7]:
print(cliente.list_database_names())

['RecursosHumanos', 'admin', 'config', 'local']


In [8]:
bill = coleccion.find_one(
    {"nombre": "Bill Gates"},
    {"_id": 0, "nombre": 1, "titular": 1}
)

print(bill)

{'nombre': 'Bill Gates', 'titular': 'Greater Seattle Area | Philanthropy'}


In [9]:
resultados = coleccion.find(
    {"experiencia.empresa": "Linux Foundation"},
    {"_id": 0, "nombre": 1}
)

for persona in resultados:
    print(persona)

{'nombre': 'Linus Torvalds'}


In [10]:
resultados = coleccion.find(
    {"formacion.inicio": 1973},
    {"_id": 0, "nombre": 1, "formacion": 1}
)

for persona in resultados:
    print(persona)

{'nombre': 'Bill Gates', 'formacion': [{'institucion': 'Harvard University', 'inicio': 1973, 'fin': 1975}, {'institucion': 'Lakeside School, Seattle', 'inicio': 1967, 'fin': 1973}]}


In [11]:
resultados = coleccion.find(
    {"formacion.inicio": {"$gte": 1980, "$lte": 1990}},
    {"_id": 0, "nombre": 1, "formacion": 1}
)

for persona in resultados:
    print("Nombre:", persona["nombre"])
    
    for formacion in persona["formacion"]:
        if 1980 <= formacion["inicio"] <= 1990:
            print("Institución:", formacion["institucion"])
    
    print("-" * 50)

Nombre: Steve Wozniak
Institución: UC Berkeley
--------------------------------------------------
Nombre: Linus Torvalds
Institución: University of Helsinki
--------------------------------------------------


In [12]:
resultado = coleccion.update_one(
    {"nombre": "Steve Wozniak"},
    {"$set": {"resumen": "Silicon Valley | Tech Speaker & Legend"}}
)

print("Documentos modificados:", resultado.modified_count)

Documentos modificados: 1


In [13]:
resultado = coleccion.delete_one(
    {"nombre": "Linus Torvalds"}
)

print("Documentos eliminados:", resultado.deleted_count)

Documentos eliminados: 1


In [14]:
for curriculum in coleccion.find({}, {"_id": 0, "nombre": 1}):
    print(curriculum)

{'nombre': 'Bill Gates'}
{'nombre': 'Steve Wozniak'}


In [15]:
total = coleccion.count_documents({})

print("Número de documentos:", total)

Número de documentos: 2


In [16]:
indice = coleccion.create_index("formacion.inicio")

print("Índice creado:", indice)


Índice creado: formacion.inicio_1


In [17]:
for indice in coleccion.list_indexes():
    print(indice)

SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_'), ('collation', SON([('locale', 'simple')]))])
SON([('v', 2), ('key', SON([('formacion.inicio', 1)])), ('name', 'formacion.inicio_1'), ('collation', SON([('locale', 'simple')]))])


In [18]:
for persona in coleccion.find(
    {},
    {"_id": 0, "nombre": 1, "titular": 1}
).sort("nombre", 1):
    print(persona)

{'nombre': 'Bill Gates', 'titular': 'Greater Seattle Area | Philanthropy'}
{'nombre': 'Steve Wozniak', 'titular': 'Silicon Valley | Hardware & Computing'}
